# 03 - Parameter Sensitivity Analysis

**Purpose:**
- Address Editor Comment C2 by proving model stability across hyperparameter choices.
- Sweep Isolation Forest over `n_estimators` and `contamination`.
- Sweep Local Outlier Factor over `n_neighbors` and `contamination`.
- Compute pairwise Jaccard Index across grid points to demonstrate that the anomalies found are stable and not artifacts of a specific parameter setting.

**Inputs:**
- `data/processed/features_raw.parquet`

**Outputs:**
- `outputs/tables/table_sensitivity_if.csv`
- `outputs/tables/table_sensitivity_lof.csv`
- `outputs/figures/fig_sensitivity_heatmap.png`\n

In [ ]:
# Cell 01: Mount Storage & Bootstrap Paths
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Paper1_Revision')
else:
    BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

for folder in ['data/raw', 'data/interim', 'data/processed', 
               'outputs/figures', 'outputs/tables', 'outputs/models', 
               'outputs/notebook_exports']:
    (BASE_DIR / folder).mkdir(parents=True, exist_ok=True)
    
print(f"Base directory set to: {BASE_DIR}")\n

In [ ]:
# Cell 02: Imports, Global Seeds & Style
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import jaccard_score
import warnings
import itertools

warnings.filterwarnings('ignore')

plt.style.use('default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'figure.dpi': 300,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.autolayout': True
})\n

In [ ]:
# Cell 03: Load Processed Features and Scale
input_path = BASE_DIR / 'data' / 'processed' / 'features_raw.parquet'
df_features = pd.read_parquet(input_path)
print(f"Data shape: {df_features.shape}")

# Scale exactly as in NB02
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features)\n

In [ ]:
# Cell 04: Define Parameter Grids
# We bracket the chosen 0.05 contamination rate with lower and higher plausible bounds
contaminations = [0.01, 0.03, 0.05, 0.07, 0.10]

# Isolation Forest Grid
if_n_estimators = [50, 100, 200, 300]

# Local Outlier Factor Grid
lof_n_neighbors = [10, 20, 30, 50]\n

In [ ]:
# Cell 05: Sweep Isolation Forest
if_results = []
if_labels_dict = {}

print("Running Isolation Forest Sensitivity Sweep...")
for n_est, cont in itertools.product(if_n_estimators, contaminations):
    model = IsolationForest(
        n_estimators=n_est, 
        contamination=cont, 
        random_state=42, 
        n_jobs=-1
    )
    labels = model.fit_predict(X_scaled)
    
    # Store labels to compute pairwise Jaccard later
    config_name = f"n{n_est}_c{cont}"
    if_labels_dict[config_name] = labels
    
    anomaly_count = (labels == -1).sum()
    if_results.append({
        'n_estimators': n_est,
        'contamination': cont,
        'anomaly_count': anomaly_count
    })

df_if_sensitivity = pd.DataFrame(if_results)
df_if_sensitivity_pivot = df_if_sensitivity.pivot(index='n_estimators', columns='contamination', values='anomaly_count')

print("\nIsolation Forest Anomaly Counts:")
display(df_if_sensitivity_pivot)

# Export
out_if = BASE_DIR / 'outputs' / 'tables' / 'table_sensitivity_if.csv'
df_if_sensitivity_pivot.to_csv(out_if)
print(f"Exported to {out_if}")\n

In [ ]:
# Cell 06: Sweep Local Outlier Factor
lof_results = []
lof_labels_dict = {}

print("Running Local Outlier Factor Sensitivity Sweep...")
for n_neigh, cont in itertools.product(lof_n_neighbors, contaminations):
    model = LocalOutlierFactor(
        n_neighbors=n_neigh, 
        contamination=cont, 
        n_jobs=-1
    )
    labels = model.fit_predict(X_scaled)
    
    config_name = f"k{n_neigh}_c{cont}"
    lof_labels_dict[config_name] = labels
    
    anomaly_count = (labels == -1).sum()
    lof_results.append({
        'n_neighbors': n_neigh,
        'contamination': cont,
        'anomaly_count': anomaly_count
    })

df_lof_sensitivity = pd.DataFrame(lof_results)
df_lof_sensitivity_pivot = df_lof_sensitivity.pivot(index='n_neighbors', columns='contamination', values='anomaly_count')

print("\nLOF Anomaly Counts:")
display(df_lof_sensitivity_pivot)

# Export
out_lof = BASE_DIR / 'outputs' / 'tables' / 'table_sensitivity_lof.csv'
df_lof_sensitivity_pivot.to_csv(out_lof)
print(f"Exported to {out_lof}")\n

In [ ]:
# Cell 07: Compute Jaccard Stability for the chosen configurations
# We compare how stable the anomaly sets are when we vary parameters slightly around our chosen defaults:
# IF default: n_estimators=100, contamination=0.05
# LOF default: n_neighbors=20, contamination=0.05

def compute_jaccard(y_true, y_pred):
    # Map -1 (anomaly) to 1, and 1 (normal) to 0 for standard jaccard_score behavior
    y1 = (y_true == -1).astype(int)
    y2 = (y_pred == -1).astype(int)
    # The intersection over union of anomalies
    intersection = (y1 & y2).sum()
    union = (y1 | y2).sum()
    return intersection / union if union > 0 else 1.0

# 1. IF Stability against n_estimators (holding contamination=0.05)
if_default = if_labels_dict['n100_c0.05']
if_jaccard = []
for n_est in if_n_estimators:
    score = compute_jaccard(if_default, if_labels_dict[f'n{n_est}_c0.05'])
    if_jaccard.append({'Varied Parameter': f'IF n_estimators={n_est}', 'Jaccard vs Default': score})

# 2. LOF Stability against n_neighbors (holding contamination=0.05)
lof_default = lof_labels_dict['k20_c0.05']
lof_jaccard = []
for k in lof_n_neighbors:
    score = compute_jaccard(lof_default, lof_labels_dict[f'k{k}_c0.05'])
    lof_jaccard.append({'Varied Parameter': f'LOF n_neighbors={k}', 'Jaccard vs Default': score})

df_stability = pd.concat([pd.DataFrame(if_jaccard), pd.DataFrame(lof_jaccard)], ignore_index=True)
print("\nStability Analysis (Jaccard Index vs Chosen Default):")
display(df_stability)

# Export Table
out_stab = BASE_DIR / 'outputs' / 'tables' / 'table_stability_jaccard.csv'
df_stability.to_csv(out_stab, index=False)\n

In [ ]:
# Cell 08: Visualizing Stability (Heatmap)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot IF Anomaly Counts
sns.heatmap(df_if_sensitivity_pivot, annot=True, fmt="d", cmap="YlGnBu", ax=axes[0])
axes[0].set_title('Isolation Forest: Anomaly Count by Parameters')
axes[0].set_xlabel('Contamination')
axes[0].set_ylabel('Number of Estimators')

# Plot LOF Anomaly Counts
sns.heatmap(df_lof_sensitivity_pivot, annot=True, fmt="d", cmap="YlGnBu", ax=axes[1])
axes[1].set_title('LOF: Anomaly Count by Parameters')
axes[1].set_xlabel('Contamination')
axes[1].set_ylabel('Number of Neighbors')

plt.tight_layout()

# Export figure
fig_heatmap = BASE_DIR / 'outputs' / 'figures' / 'fig_sensitivity_heatmap.png'
plt.savefig(fig_heatmap, dpi=300, bbox_inches='tight')
print(f"Saved Sensitivity Heatmap to {fig_heatmap}")
plt.show()\n